# 1. The card database: ingest, schema, and faces

Everything upstream of the game model. Where 35,000 cards come from, how they are
stored, and the one structural decision that caused the most bugs — **faces**.

The whole layer is keyless and offline: a public Scryfall bulk export downloaded
once into DuckDB. `data/` is a build artifact, rebuildable in one command, never
committed and never hand-edited.

    manalab fetch      # download the Scryfall oracle_cards bulk export
    manalab build      # load it into data/cards.duckdb
    manalab info       # what is in there

In [1]:
from pathlib import Path

import duckdb

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

DB = REPO / "data" / "cards.duckdb"
con = duckdb.connect(str(DB), read_only=True)


def q(sql, params=None, limit=12):
    # Run a query and print it as a small table.
    rows = con.execute(sql, params or []).fetchall()
    names = [d[0] for d in con.description]
    shown = rows[:limit]
    if not shown:
        print("  ".join(map(str, names)))
        print("(no rows)")
        return rows
    widths = [max([len(str(n))] + [len(str(r[i])[:38]) for r in shown])
              for i, n in enumerate(names)]
    print("  ".join(str(n).ljust(w) for n, w in zip(names, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v)[:38].ljust(w) for v, w in zip(row, widths)))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")
    return rows

## Five tables

Two hold what Scryfall said. Three are derived, and exist because the raw shape
is awkward to ask questions of.

In [2]:
for table in ["cards", "faces", "face_types", "tags", "card_tags"]:
    n = con.execute(f"select count(*) from {table}").fetchone()[0]
    cols = [r[0] for r in con.execute(
        "select column_name from information_schema.columns where table_name = ?",
        [table]).fetchall()]
    print(f"{table:<12} {n:>9,} rows")
    print(f"             {', '.join(cols)}\n")

cards           35,142 rows
             oracle_id, name, mana_cost, cmc, type_line, oracle_text, layout, colors, color_identity, produced_mana, legalities, rarity, set_name, power, toughness, loyalty, keywords

faces           36,033 rows
             oracle_id, face_index, name, mana_cost, type_line, oracle_text, power, toughness, loyalty

face_types      36,033 rows
             oracle_id, face_index, supertypes, card_types, subtypes, unknown, is_permanent

tags             4,520 rows
             slug, label, description, parent_ids, card_count

card_tags      229,411 rows
             oracle_id, tag, weight



## `cards` — one row per card

Note what is *not* normalised: `colors`, `keywords`, `legalities` and
`produced_mana` are stored as JSON. DuckDB can query into them directly, and
flattening them into join tables would buy nothing for the questions this project
asks.

In [3]:
q('''
select name, mana_cost, cmc, type_line, power, toughness, rarity
from cards
where name in ('Lightning Bolt', 'Leatherback Baloth', 'Forest', 'Tarmogoyf')
''')

name                mana_cost  cmc  type_line            power  toughness  rarity  
------------------  ---------  ---  -------------------  -----  ---------  --------
Leatherback Baloth  {G}{G}{G}  3.0  Creature — Beast     4      5          uncommon
Lightning Bolt      {R}        1.0  Instant              None   None       uncommon
Tarmogoyf           {1}{G}     2.0  Creature — Lhurgoyf  *      1+*        mythic  
Forest                         0.0  Basic Land — Forest  None   None       common  


[('Leatherback Baloth',
  '{G}{G}{G}',
  3.0,
  'Creature — Beast',
  '4',
  '5',
  'uncommon'),
 ('Lightning Bolt', '{R}', 1.0, 'Instant', None, None, 'uncommon'),
 ('Tarmogoyf', '{1}{G}', 2.0, 'Creature — Lhurgoyf', '*', '1+*', 'mythic'),
 ('Forest', '', 0.0, 'Basic Land — Forest', None, None, 'common')]

In [4]:
# JSON columns are queried in place.
q('''
select name, colors, color_identity, keywords,
       json_extract_string(legalities, '$.modern') as modern
from cards
where name in ('Birds of Paradise', 'Vampire Nighthawk', 'Noble Hierarch')
''')

name               colors  color_identity   keywords                              modern
-----------------  ------  ---------------  ------------------------------------  ------
Noble Hierarch     ['G']   ['G', 'U', 'W']  ['Exalted']                           legal 
Birds of Paradise  ['G']   ['G']            ['Flying']                            legal 
Vampire Nighthawk  ['B']   ['B']            ['Deathtouch', 'Flying', 'Lifelink']  legal 


[('Noble Hierarch', ['G'], ['G', 'U', 'W'], ['Exalted'], 'legal'),
 ('Birds of Paradise', ['G'], ['G'], ['Flying'], 'legal'),
 ('Vampire Nighthawk',
  ['B'],
  ['B'],
  ['Deathtouch', 'Flying', 'Lifelink'],
  'legal')]

### Power and toughness are strings, deliberately

Magic has `*` and `1+*`. Storing power as an integer forces a lie at ingest time.
The string is kept, and `Card.base_power` returns `None` rather than inventing a
number.

In [5]:
q('''
select name, power, toughness from cards
where power in ('*', '1+*', '2+*') and cmc <= 3
order by name
''', limit=6)

name                         power  toughness
---------------------------  -----  ---------
A-Uurg, Spawn of Turg        *      5        
Abomination of Llanowar      *      *        
Adamaro, First to Desire     *      *        
Adeline, Resplendent Cathar  *      4        
Anax, Hardened in the Forge  *      3        
Archpriest of Iona           *      2        
... 84 more rows


[('A-Uurg, Spawn of Turg', '*', '5'),
 ('Abomination of Llanowar', '*', '*'),
 ('Adamaro, First to Desire', '*', '*'),
 ('Adeline, Resplendent Cathar', '*', '4'),
 ('Anax, Hardened in the Forge', '*', '3'),
 ('Archpriest of Iona', '*', '2'),
 ('Avalanche of Sector 7', '*', '3'),
 ('Awoken Nephilim', '*', '*'),
 ('Barrowgoyf', '*', '1+*'),
 ('Boneyard Mycodrax', '*', '*'),
 ('Boneyard Wurm', '*', '*'),
 ('Brotherhood Vertibird', '*', '4'),
 ('Burrowguard Mentor', '*', '*'),
 ('Callaphe, Beloved of the Sea', '*', '3'),
 ('Caller of the Hunt', '*', '*'),
 ('Cantivore', '*', '*'),
 ('Coiling Woodworm', '*', '1'),
 ('Collective Drone', '*', '2'),
 ('Common Curve Filler', '*', '*'),
 ('Cosmogoyf', '*', '1+*'),
 ("Cruel Somnophage // Can't Wake Up", '*', '*'),
 ('Crusader of Odric', '*', '*'),
 ('Darksteel Hydra', '*', '*'),
 ('Dauthi Warlord', '*', '1'),
 ('Dodgy Jalopy', '*', '5'),
 ('Dragonfly Swarm', '*', '3'),
 ('Duelist of the Mind', '*', '3'),
 ('Dungrove Elder', '*', '*'),
 ('Enigma D

In [6]:
from manalab.cards.store import CardStore

store = CardStore(DB)
for name in ["Tarmogoyf", "Grizzly Bears"]:
    card = store.by_name(name)
    print(f"  {name:<16} printed {card.power}/{card.toughness}   "
          f"base_power={card.base_power}  base_toughness={card.base_toughness}")

  Tarmogoyf        printed */1+*   base_power=None  base_toughness=None


  Grizzly Bears    printed 2/2   base_power=2  base_toughness=2


## Faces: the decision that caused the most bugs

A card can have several printed sides — transforming cards, modal double-faced
cards, split cards. Scryfall puts their text on a `card_faces` array and leaves
the **top-level `oracle_text` blank**.

That blank is a trap. Code that reads `oracle_text` and finds `""` concludes
"vanilla" when the truth is "unread".

In [7]:
q('''
select name, layout, oracle_text, type_line
from cards
where layout in ('transform', 'modal_dfc', 'split')
  and (oracle_text is null or oracle_text = '')
''', limit=6)

name                                    layout     oracle_text  type_line                             
--------------------------------------  ---------  -----------  --------------------------------------
Ulvenwald Captive // Ulvenwald Abomina  transform  None         Creature — Werewolf Horror // Creature
Azusa's Many Journeys // Likeness of t  transform  None         Enchantment — Saga // Enchantment Crea
Witch Enchanter // Witch-Blessed Meado  modal_dfc  None         Creature — Human Warlock // Land      
Heaven // Earth                         split      None         Instant // Sorcery                    
The Emperor of Palamecia // The Lord M  transform  None         Legendary Creature — Human Noble Wizar
Breaking // Entering                    split      None         Sorcery // Sorcery                    
... 632 more rows


[('Ulvenwald Captive // Ulvenwald Abomination',
  'transform',
  None,
  'Creature — Werewolf Horror // Creature — Eldrazi Werewolf'),
 ("Azusa's Many Journeys // Likeness of the Seeker",
  'transform',
  None,
  'Enchantment — Saga // Enchantment Creature — Human Monk'),
 ('Witch Enchanter // Witch-Blessed Meadow',
  'modal_dfc',
  None,
  'Creature — Human Warlock // Land'),
 ('Heaven // Earth', 'split', None, 'Instant // Sorcery'),
 ('The Emperor of Palamecia // The Lord Master of Hell',
  'transform',
  None,
  'Legendary Creature — Human Noble Wizard // Legendary Creature — Demon Noble Wizard'),
 ('Breaking // Entering', 'split', None, 'Sorcery // Sorcery'),
 ('Westvale Abbey // Ormendahl, Profane Prince',
  'transform',
  None,
  'Land // Legendary Creature — Demon'),
 ('Fell the Profane // Fell Mire', 'modal_dfc', None, 'Instant // Land'),
 ('Flesh // Blood', 'split', None, 'Sorcery // Sorcery'),
 ('Lunarch Veteran // Luminous Phantom',
  'transform',
  None,
  'Creature — Human

The `faces` table fixes this: one row per printed side, with a **synthesised face
0** for single-faced cards so that nothing downstream has to branch on layout.

Note the face is parsed under its *own* name. The text of Fire says "Fire deals 2
damage", not "Fire // Ice deals 2 damage" — passing the joined name would never
match during parsing.

In [8]:
q('''
select f.name, f.face_index, f.mana_cost, f.type_line, f.oracle_text
from faces f
join cards c using (oracle_id)
where c.name like 'Bala Ged Recovery%' or c.name like 'Fire // Ice%'
order by c.name, f.face_index
''')

name                face_index  mana_cost  type_line  oracle_text                           
------------------  ----------  ---------  ---------  --------------------------------------
Bala Ged Recovery   0           {2}{G}     Sorcery    Return target card from your graveyard
Bala Ged Sanctuary  1                      Land       This land enters tapped.
{T}: Add {G}.
Fire                0           {1}{R}     Instant    Fire deals 2 damage divided as you cho
Ice                 1           {1}{U}     Instant    Tap target permanent.
Draw a card.    


[('Bala Ged Recovery',
  0,
  '{2}{G}',
  'Sorcery',
  'Return target card from your graveyard to your hand.'),
 ('Bala Ged Sanctuary',
  1,
  '',
  'Land',
  'This land enters tapped.\n{T}: Add {G}.'),
 ('Fire',
  0,
  '{1}{R}',
  'Instant',
  'Fire deals 2 damage divided as you choose among one or two targets.'),
 ('Ice', 1, '{1}{U}', 'Instant', 'Tap target permanent.\nDraw a card.')]

In [9]:
# Every card has at least one face; single-faced cards get face 0 synthesised.
q('''
select
  (select count(*) from cards) as cards,
  (select count(*) from faces) as faces,
  (select count(*) from cards c where not exists
      (select 1 from faces f where f.oracle_id = c.oracle_id)) as cards_without_a_face
''')

cards  faces  cards_without_a_face
-----  -----  --------------------
35142  36033  0                   


[(35142, 36033, 0)]

### Why this matters downstream

`engine.fidelity` decides whether a card is safe for a deck search to use. Its
first check is `card.has_faces`, *before* anything reads `oracle_text` or
`is_land` — because both mislead on a faced card.

This was a real bug: modal cards were entering the "fully modelled" pool as
textless zero-cost creatures.

In [10]:
from manalab.engine import fidelity

for name in ["Bala Ged Recovery // Bala Ged Sanctuary", "Grizzly Bears"]:
    card = store.by_name(name)
    report = fidelity.assess(card)
    print(f"  {name[:34]:<36} has_faces={card.has_faces}  "
          f"type_line={card.type_line!r}")
    print(f"      is_land says {card.is_land}, oracle_text is {card.oracle_text!r}")
    print(f"      fidelity: {report.reasons or 'fully modelled'}\n")

  Bala Ged Recovery // Bala Ged Sanc   has_faces=True  type_line='Sorcery // Land'
      is_land says True, oracle_text is ''
      fidelity: ('multi-faced; the engine plays only one side',)

  Grizzly Bears                        has_faces=False  type_line='Creature — Bear'
      is_land says False, oracle_text is ''
      fidelity: fully modelled



## `face_types` — the type line, parsed

`"Legendary Creature — Dinosaur Ooze"` is three different things: supertypes,
card types, subtypes. Parsed once at build time so that questions like "how many
creatures are there" are a `GROUP BY` rather than a `LIKE`.

Anything the parser could not classify goes in `unknown` — it refuses to guess.

In [11]:
q('''
select f.name, ft.supertypes, ft.card_types, ft.subtypes, ft.is_permanent
from face_types ft join faces f using (oracle_id, face_index)
where f.name in ('Terrian, World Tyrant', 'Lightning Bolt', 'Dryad Arbor',
                 'Forest', 'Gideon Blackblade')
''')

name                   supertypes     card_types            subtypes              is_permanent
---------------------  -------------  --------------------  --------------------  ------------
Lightning Bolt         []             ['instant']           []                    False       
Lightning Bolt         []             ['instant']           []                    False       
Gideon Blackblade      ['legendary']  ['planeswalker']      ['Gideon']            True        
Terrian, World Tyrant  ['legendary']  ['creature']          ['Dinosaur', 'Ooze']  True        
Forest                 ['basic']      ['land']              ['Forest']            True        
Dryad Arbor            []             ['land', 'creature']  ['Forest', 'Dryad']   True        


[('Lightning Bolt', [], ['instant'], [], False),
 ('Lightning Bolt', [], ['instant'], [], False),
 ('Gideon Blackblade', ['legendary'], ['planeswalker'], ['Gideon'], True),
 ('Terrian, World Tyrant',
  ['legendary'],
  ['creature'],
  ['Dinosaur', 'Ooze'],
  True),
 ('Forest', ['basic'], ['land'], ['Forest'], True),
 ('Dryad Arbor', [], ['land', 'creature'], ['Forest', 'Dryad'], True)]

In [12]:
# What refusing to guess looks like: nine unknowns in 36,033 faces, all jokes.
q('''
select f.name, ft.unknown, f.type_line
from face_types ft join faces f using (oracle_id, face_index)
where ft.unknown != '[]' and ft.unknown is not null
''', limit=10)

name                        unknown                                type_line                        
--------------------------  -------------------------------------  ---------------------------------
Atinlay Igpay               ['Eaturecray']                         Eaturecray — Igpay               
Trial and Error             ['Elemental']                          Elemental Instant — Fire         
That's Enough Slices        ['Phenome-nom']                        Phenome-nom                      
Byode, Inverse Sun          ['Universewalker']                     Legendary Universewalker — Byode 
B.F.M. (Big Furry Monster)  ['Scariest', "You'll", 'Ever', 'See']  Scariest Creature You'll Ever See
Orb of Origin               ['Poly']                               Poly Artifact                    


[('Atinlay Igpay', ['Eaturecray'], 'Eaturecray — Igpay'),
 ('Trial and Error', ['Elemental'], 'Elemental Instant — Fire'),
 ("That's Enough Slices", ['Phenome-nom'], 'Phenome-nom'),
 ('Byode, Inverse Sun',
  ['Universewalker'],
  'Legendary Universewalker — Byode'),
 ('B.F.M. (Big Furry Monster)',
  ['Scariest', "You'll", 'Ever', 'See'],
  "Scariest Creature You'll Ever See"),
 ('Orb of Origin', ['Poly'], 'Poly Artifact')]

## Verification: the corpus is checked against Scryfall itself

`examples/verify_exact.py` walks every card, every face, every field and compares
against Scryfall's own export. Zero diffs is the contract, and its exit code is
the test.

Downloads also record `updated_at` and a sha256 in a sidecar, and the skip
compares against *that* rather than against the file merely existing — which is
how a three-day-stale file once hid behind a "done" message.

In [13]:
import json

meta = json.loads((REPO / "data" / "oracle_cards.jsonl.gz.meta.json").read_text())
for key, value in meta.items():
    print(f"  {key:<14} {str(value)[:70]}")

  kind           oracle_cards
  updated_at     2026-08-11T09:01:55.863+00:00
  download_uri   https://data.scryfall.io/oracle-cards/oracle-cards-20260811090155.json
  sha256         04fe675b955b33fab9beaba604d459e6b42771bc35f9779e193dfc0a95c6a91b
  bytes          24502785


## What to take into notebook 2

- Five tables: `cards` and `faces` are Scryfall's; `face_types`, `tags` and
  `card_tags` are derived.
- Power/toughness stay strings because `*` exists; `base_power` returns `None`.
- **Faced cards have blank top-level text.** Blank means *unread*, not vanilla —
  check `has_faces` before trusting any text-based rule.
- Type lines are parsed into supertypes/types/subtypes at build time, with
  unknowns recorded rather than guessed.

Next: turning oracle text into effect verbs.